In [1]:
import pandas as pd

databases

In [2]:
orders = '../databases/orders.csv'
order_items = '../databases/order_items.csv'
products = '../databases/products.csv'
users = '../databases/users.csv'

read csv

In [3]:
orders_df = pd.read_csv(orders)
order_items_df = pd.read_csv(order_items)
users_df = pd.read_csv(users)

C:\Users\User\AppData\Local\Temp\ipykernel_10220\1125601531.py:3: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  users_df = pd.read_csv(users)


convert datetime

In [4]:
orders_df["created_at"] = pd.to_datetime(
    orders_df["created_at"],
    format="ISO8601"
)

JOIN orders + order_items

In [5]:
merged_df = orders_df.merge(
    order_items_df,
    on="order_id",
    how="inner"
)

merged_df

,order_id,user_id_x,status_x,gender,created_at_x,returned_at_x,shipped_at_x,delivered_at_x,num_of_item,id,user_id_y,product_id,inventory_item_id,status_y,created_at_y,shipped_at_y,delivered_at_y,returned_at_y,sale_price
0,8,8,Cancelled,F,2023-07-22 08:23:24+00:00,NaN,NaN,NaN,1,13,8,1980,36,Cancelled,2023-07-22 05:33:30+00:00,NaN,NaN,NaN,79.000000
1,29,29,Cancelled,F,2026-02-10 06:15:05+00:00,NaN,NaN,NaN,1,47,29,10707,124,Cancelled,2026-02-10 03:05:51+00:00,NaN,NaN,NaN,11.900000
2,60,59,Cancelled,F,2025-06-07 05:45:33+00:00,NaN,NaN,NaN,1,96,59,12709,265,Cancelled,2025-06-07 05:44:37+00:00,NaN,NaN,NaN,79.000000
3,65,62,Cancelled,F,2026-05-18 07:09:09+00:00,NaN,NaN,NaN,1,101,62,5964,280,Cancelled,2026-05-18 06:41:04+00:00,NaN,NaN,NaN,21.990000
4,68,63,Cancelled,F,2023-11-25 02:08:33+00:00,NaN,NaN,NaN,1,104,63,15394,288,Cancelled,2023-11-24 23:47:00+00:00,NaN,NaN,NaN,22.990000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181514,125417,99997,Shipped,M,2022-05-15 22:42:57+00:00,NaN,2022-05-15 23:08:57+00:00,NaN,1,181512,99997,18818,489645,Shipped,2022-05-15 21:25:32+00:00,2022-05-15 23:08:57+00:00,NaN,NaN,45.990002
181515,125418,99997,Shipped,M,2025-02-17 08:47:30+00:00,NaN,2025-02-18 17:29:30+00:00,NaN,1,181513,99997,24870,489649,Shipped,2025-02-17 08:14:15+00:00,2025-02-18 17:29:30+00:00,NaN,NaN,20.950001
181516,125419,99998,Shipped,M,2024-02-16 08:43:15+00:00,NaN,2024-02-17 03:31:15+00:00,NaN,3,181515,99998,28912,489654,Shipped,2024-02-19 07:35:34+00:00,2024-02-17 03:31:15+00:00,NaN,NaN,14.950000
181517,125419,99998,Shipped,M,2024-02-16 08:43:15+00:00,NaN,2024-02-17 03:31:15+00:00,NaN,3,181514,99998,20636,489652,Shipped,2024-02-20 07:12:43+00:00,2024-02-17 03:31:15+00:00,NaN,NaN,29.990000


JOIN users

In [6]:
merged_df = merged_df.merge(
    users_df,
    left_on="user_id_x",
    right_on="id",
    how="inner"
)

condition WHERE status not cancelled or returned

In [7]:
filtered_df = merged_df[
    ~merged_df["status_x"].isin(["Cancelled", "Returned"])
]

Group by user_id and country

In [8]:
customer_summary = (
    filtered_df
    .groupby(["user_id_x", "country"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_revenue=("sale_price", "sum"),
        first_order_date=("created_at", "min"),
        last_order_date=("created_at", "max")
    )
)

Active months

In [9]:
customer_summary["last_order_date"] = pd.to_datetime(
    customer_summary["last_order_date"],
    format="ISO8601"
)

customer_summary["first_order_date"] = pd.to_datetime(
    customer_summary["first_order_date"],
    format="ISO8601"
)

customer_summary["active_months"] = (
    (
        customer_summary["last_order_date"].dt.year
        - customer_summary["first_order_date"].dt.year
    ) * 12
    +
    (
        customer_summary["last_order_date"].dt.month
        - customer_summary["first_order_date"].dt.month
    )
    + 1
)
customer_summary

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months
0,2,Brasil,2,232.450001,2024-06-11 08:23:00+00:00,2024-06-11 08:23:00+00:00,1
1,5,China,1,258.000000,2023-04-29 06:24:00+00:00,2023-04-29 06:24:00+00:00,1
2,6,Brasil,1,16.879999,2024-08-14 12:37:00+00:00,2024-08-14 12:37:00+00:00,1
3,7,Brasil,1,43.570002,2024-11-22 06:14:00+00:00,2024-11-22 06:14:00+00:00,1
4,9,Brasil,2,162.740000,2020-02-20 02:35:00+00:00,2020-02-20 02:35:00+00:00,1
...,...,...,...,...,...,...,...
66118,99996,Belgium,1,158.000000,2022-04-16 12:20:00+00:00,2022-04-16 12:20:00+00:00,1
66119,99997,Brasil,2,66.940002,2020-06-22 12:34:00+00:00,2020-06-22 12:34:00+00:00,1
66120,99998,United States,2,439.879987,2025-08-30 17:34:00+00:00,2025-08-30 17:34:00+00:00,1
66121,99999,Brasil,1,20.000000,2024-10-19 14:33:00+00:00,2024-10-19 14:33:00+00:00,1


Keep only date part

In [10]:
customer_summary["last_order_date"] = pd.to_datetime(
    customer_summary["last_order_date"],
    format="ISO8601"
)

customer_summary["first_order_date"] = pd.to_datetime(
    customer_summary["first_order_date"],
    format="ISO8601"
)

customer_summary["first_order_date"] = (
    customer_summary["first_order_date"].dt.date
)

customer_summary["last_order_date"] = (
    customer_summary["last_order_date"].dt.date
)

LTV

In [11]:
customer_summary["ltv"] = (
    customer_summary["total_revenue"]
    .round(2)
)

customer_summary

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv
0,2,Brasil,2,232.450001,2024-06-11,2024-06-11,1,232.45
1,5,China,1,258.000000,2023-04-29,2023-04-29,1,258.00
2,6,Brasil,1,16.879999,2024-08-14,2024-08-14,1,16.88
3,7,Brasil,1,43.570002,2024-11-22,2024-11-22,1,43.57
4,9,Brasil,2,162.740000,2020-02-20,2020-02-20,1,162.74
...,...,...,...,...,...,...,...,...
66118,99996,Belgium,1,158.000000,2022-04-16,2022-04-16,1,158.00
66119,99997,Brasil,2,66.940002,2020-06-22,2020-06-22,1,66.94
66120,99998,United States,2,439.879987,2025-08-30,2025-08-30,1,439.88
66121,99999,Brasil,1,20.000000,2024-10-19,2024-10-19,1,20.00


Average Order Value

In [12]:
customer_summary["avg_order_value"] = (
    customer_summary["total_revenue"]
    / customer_summary["total_orders"]
).round(2)
customer_summary

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv,avg_order_value
0,2,Brasil,2,232.450001,2024-06-11,2024-06-11,1,232.45,116.23
1,5,China,1,258.000000,2023-04-29,2023-04-29,1,258.00,258.00
2,6,Brasil,1,16.879999,2024-08-14,2024-08-14,1,16.88,16.88
3,7,Brasil,1,43.570002,2024-11-22,2024-11-22,1,43.57,43.57
4,9,Brasil,2,162.740000,2020-02-20,2020-02-20,1,162.74,81.37
...,...,...,...,...,...,...,...,...,...
66118,99996,Belgium,1,158.000000,2022-04-16,2022-04-16,1,158.00,158.00
66119,99997,Brasil,2,66.940002,2020-06-22,2020-06-22,1,66.94,33.47
66120,99998,United States,2,439.879987,2025-08-30,2025-08-30,1,439.88,219.94
66121,99999,Brasil,1,20.000000,2024-10-19,2024-10-19,1,20.00,20.00


Average Monthly Value

In [13]:
customer_summary["avg_monthly_revenue"] = (
    customer_summary["total_revenue"]
    / customer_summary["active_months"]
).round(2)
customer_summary

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv,avg_order_value,avg_monthly_revenue
0,2,Brasil,2,232.450001,2024-06-11,2024-06-11,1,232.45,116.23,232.45
1,5,China,1,258.000000,2023-04-29,2023-04-29,1,258.00,258.00,258.00
2,6,Brasil,1,16.879999,2024-08-14,2024-08-14,1,16.88,16.88,16.88
3,7,Brasil,1,43.570002,2024-11-22,2024-11-22,1,43.57,43.57,43.57
4,9,Brasil,2,162.740000,2020-02-20,2020-02-20,1,162.74,81.37,162.74
...,...,...,...,...,...,...,...,...,...,...
66118,99996,Belgium,1,158.000000,2022-04-16,2022-04-16,1,158.00,158.00,158.00
66119,99997,Brasil,2,66.940002,2020-06-22,2020-06-22,1,66.94,33.47,66.94
66120,99998,United States,2,439.879987,2025-08-30,2025-08-30,1,439.88,219.94,439.88
66121,99999,Brasil,1,20.000000,2024-10-19,2024-10-19,1,20.00,20.00,20.00


LTV quartile

In [14]:
customer_summary["ltv_quartile"] = pd.qcut(
    customer_summary["total_revenue"],
    4,
    labels=[1, 2, 3, 4]
)
customer_summary

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv,avg_order_value,avg_monthly_revenue,ltv_quartile
0,2,Brasil,2,232.450001,2024-06-11,2024-06-11,1,232.45,116.23,232.45,4
1,5,China,1,258.000000,2023-04-29,2023-04-29,1,258.00,258.00,258.00,4
2,6,Brasil,1,16.879999,2024-08-14,2024-08-14,1,16.88,16.88,16.88,1
3,7,Brasil,1,43.570002,2024-11-22,2024-11-22,1,43.57,43.57,43.57,2
4,9,Brasil,2,162.740000,2020-02-20,2020-02-20,1,162.74,81.37,162.74,3
...,...,...,...,...,...,...,...,...,...,...,...
66118,99996,Belgium,1,158.000000,2022-04-16,2022-04-16,1,158.00,158.00,158.00,3
66119,99997,Brasil,2,66.940002,2020-06-22,2020-06-22,1,66.94,33.47,66.94,2
66120,99998,United States,2,439.879987,2025-08-30,2025-08-30,1,439.88,219.94,439.88,4
66121,99999,Brasil,1,20.000000,2024-10-19,2024-10-19,1,20.00,20.00,20.00,1


Ordering by LTV (desc)

In [15]:
customer_ltv = customer_summary.sort_values(
    "ltv",
    ascending=False
)

customer_ltv

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv,avg_order_value,avg_monthly_revenue,ltv_quartile
38808,58793,United States,4,1850.630014,2020-07-29,2020-07-29,1,1850.63,462.66,1850.63,4
59666,90351,China,3,1701.970001,2023-02-21,2023-02-21,1,1701.97,567.32,1701.97,4
51913,78596,China,4,1504.370001,2025-01-22,2025-01-22,1,1504.37,376.09,1504.37,4
12013,18218,China,3,1479.480011,2019-10-07,2019-10-07,1,1479.48,493.16,1479.48,4
58742,88969,Spain,4,1467.650002,2024-05-22,2024-05-22,1,1467.65,366.91,1467.65,4
...,...,...,...,...,...,...,...,...,...,...,...
49351,74672,Brasil,1,1.720000,2026-02-18,2026-02-18,1,1.72,1.72,1.72,1
50430,76338,Belgium,1,1.720000,2026-04-16,2026-04-16,1,1.72,1.72,1.72,1
53624,81231,Germany,1,1.500000,2020-01-11,2020-01-11,1,1.50,1.50,1.50,1
23159,34967,Spain,1,1.500000,2022-07-09,2022-07-09,1,1.50,1.50,1.50,1


Active months should be >= 1

In [16]:
customer_summary[
    customer_summary["active_months"] < 1
]

,user_id_x,country,total_orders,total_revenue,first_order_date,last_order_date,active_months,ltv,avg_order_value,avg_monthly_revenue,ltv_quartile
